## 1. Environment Setup and Library Imports

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# -------------------------------------------------------------------------
# Path resolution to access root directory and 'src' modules
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import local custom modules
from src.pkldataset import PKLDataset
from src.helpers import set_seed, get_model, train_model, eval_model

# Hardware configuration and loss function
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion = nn.CrossEntropyLoss()

# Random seeds for evaluation (10 seeds matching benchmarks)
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# Base sheet thickness used for HC training subsets
source_thickness = "T197"

# -------------------------------------------------------------------------
# Dataset Paths & Configurations (Late Fusion: HC T197 -> HC Targets)
# -------------------------------------------------------------------------
# Se apunta a la carpeta RP y PKLDataset empareja internamente con AZ (modality="fusion")
train_path_hc = str(DATA_DIR / "HC" / source_thickness / "RP")

sample_sizes = [20, 50, 100, 200, 300, 400, 500]
subset_names = [f"train_{size}" for size in sample_sizes]

# Rutas de evaluación: los 4 espesores restantes de HC (100% de muestras)
val_paths = [
    str(DATA_DIR / "HC" / "T185" / "RP"),
    str(DATA_DIR / "HC" / "T188" / "RP"),
    str(DATA_DIR / "HC" / "T191" / "RP"),
    str(DATA_DIR / "HC" / "T194" / "RP"),
]

# Cargar dataset completo de HC T197 en modo fusion
full_hc_dataset = PKLDataset(folder_path=train_path_hc, modality="fusion")

# Contenedor de resultados: results[subset_name][val_path] -> lista de precisiones
results = {s_name: {vp: [] for vp in val_paths} for s_name in subset_names}

# -------------------------------------------------------------------------
# Verificación de rutas
# -------------------------------------------------------------------------
print("Verifying folder paths for Late Fusion (HC -> HC)...")
for p in [train_path_hc] + val_paths:
    exists = os.path.exists(p)
    print(f"Path ({p}): {'FOUND' if exists else 'MISSING'}")

print(f"\nHC Source Training Path (Fusion): {train_path_hc}")
print(f"HC Total Available Samples: {len(full_hc_dataset)}")
print(f"HC Training Subsets to Evaluate: {subset_names}")
print(f"HC Target Evaluation Thicknesses: {[Path(vp).parent.name for vp in val_paths]}")
print(f"Total Random Seeds: {len(seeds)}")

## 2. Configuration (Fusion)

In [ ]:
import sys
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import DataLoader, Subset

# -------------------------------------------------------------------------
# Path resolution
# -------------------------------------------------------------------------
PROJECT_ROOT = Path.cwd().parent if "__file__" not in globals() else Path(__file__).resolve().parent.parent
DATA_DIR = PROJECT_ROOT / "data"

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.pkldataset import PKLDataset, class_to_onehot

# -------------------------------------------------------------------------
# Stratified Sampling Helper for HC Subsets
# -------------------------------------------------------------------------
def get_balanced_transfer_subset(dataset, total_samples, seed=42):
    """
    Extracts an exactly class-balanced subset from a PKLDataset or Subset instance.
    For example: 20 samples -> 2 samples per wear class (R05..R50).
    Ensures indices are native integers to avoid collation issues in DataLoader.
    """
    np.random.seed(seed)
    num_classes = 10
    samples_per_class = total_samples // num_classes
    
    if isinstance(dataset, Subset):
        base_dataset = dataset.dataset
        available_indices = list(dataset.indices)
    else:
        base_dataset = dataset
        available_indices = list(range(len(dataset.files)))
    
    # Map subset indices by wear class (R05 to R50)
    class_indices = {i: [] for i in range(num_classes)}
    for idx in available_indices:
        filename = base_dataset.files[int(idx)]
        class_label = filename.split('_')[0]
        c_idx = int(np.argmax(class_to_onehot[class_label]))
        class_indices[c_idx].append(int(idx))
        
    selected_indices = []
    for c_idx, indices in class_indices.items():
        if len(indices) < samples_per_class:
            raise ValueError(f"No hay suficientes muestras en la clase {c_idx}: {len(indices)} disponibles vs {samples_per_class} pedidas.")
        chosen = np.random.choice(indices, size=samples_per_class, replace=False)
        selected_indices.extend([int(x) for x in chosen])
        
    return Subset(base_dataset, selected_indices)

# -------------------------------------------------------------------------
# Dataset Paths & Configurations (Late Fusion: HC T197 -> Other HC Targets)
# -------------------------------------------------------------------------
# Source domain training base path: HC T197 (pairs RP and AZ automatically)
source_thickness = "T197"
train_path_hc = str(DATA_DIR / "HC" / source_thickness / "RP")

# Sample sizes extracted from HC (T197) to train models from scratch
sample_sizes = [20, 50, 100, 200, 300, 400, 500]
subset_names = [f"train_{size}" for size in sample_sizes]

# Evaluation paths: independent HC sheet thicknesses (100% of samples)
val_paths = [
    str(DATA_DIR / "HC" / "T185" / "RP"),
    str(DATA_DIR / "HC" / "T188" / "RP"),
    str(DATA_DIR / "HC" / "T191" / "RP"),
    str(DATA_DIR / "HC" / "T194" / "RP"),
]

# Random seeds for reproducibility across runs (10 seeds)
seeds = [101, 202, 303, 404, 505, 606, 707, 808, 909, 1001]

# Load full HC source dataset in fusion mode
full_hc_dataset = PKLDataset(folder_path=train_path_hc, modality="fusion")

# Results container for subset evaluations: results[subset_name][val_path] -> list of accuracies
results = {s_name: {vp: [] for vp in val_paths} for s_name in subset_names}

## 3. Validation Preloading - Fusion

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from src.pkldataset import PKLDataset

# -------------------------------------------------------------------------
# Preload full HC evaluation datasets into RAM (Fusion: 2 channels)
# -------------------------------------------------------------------------
print("Preloading full HC evaluation datasets into RAM (Fusion: 2 channels)...")

# Asignar workers de CPU de forma segura (4 a 8 es ideal para I/O de archivos)
cpu_cores = os.cpu_count() or 4
num_workers = min(8, cpu_cores)
cached_val_tensors = {}

for vp in val_paths:
    thickness_name = vp.split("/")[-2] if "/" in vp else vp.split("\\")[-2]
    print(f"  -> Caching HC_{thickness_name}...", end=" ", flush=True)

    temp_ds = PKLDataset(folder_path=vp, modality="fusion")
    
    # num_workers > 0 paraleliza la apertura y deserialización de los PKL
    temp_loader = DataLoader(
        temp_ds,
        batch_size=512,            # Lote más grande para reducir overhead de colación
        shuffle=False,
        num_workers=num_workers,
        pin_memory=False,          # Mantener en False durante el caching en RAM principal
        persistent_workers=False
    )
    
    all_inputs, all_targets = [], []
    for x, y in temp_loader:
        all_inputs.append(x)
        all_targets.append(y)
        
    cached_val_tensors[vp] = {
        "inputs": torch.cat(all_inputs, dim=0),
        "targets": torch.cat(all_targets, dim=0)
    }
    
    n_samples = cached_val_tensors[vp]["inputs"].shape[0]
    print(f"Listo ({n_samples} muestras cargadas).")

print("\nHC evaluation datasets successfully preloaded into RAM.")
print(f"HC Source Training Path (Fusion): {train_path_hc}")
print(f"HC Total Available Samples: {len(full_hc_dataset)}")
print(f"HC Training Subsets to Evaluate: {subset_names}")
print(f"HC Evaluation Paths (Fusion): {val_paths}")
print(f"Total Random Seeds: {len(seeds)}")

## 4. Training and Evaluation - Fusion

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

# -------------------------------------------------------------------------
# Late Fusion Architecture (Robust Dynamic Feature Extraction)
# -------------------------------------------------------------------------
class LateFusion1DCNN(nn.Module):
    def __init__(self, sample_length, num_classes=10):
        super().__init__()
        # Instanciar modelos base independientes de 1 canal
        self.branch_rp = get_model(input_channels=1)
        self.branch_az = get_model(input_channels=1)

        # Desactivar la cabeza clasificadora original en ambas ramas
        for branch in (self.branch_rp, self.branch_az):
            if hasattr(branch, "fc"):
                branch.fc = nn.Identity()
            elif hasattr(branch, "classifier"):
                branch.classifier = nn.Identity()

        # Determinar la dimensión exacta de características usando la longitud real de la señal
        with torch.no_grad():
            self.branch_rp.eval()
            dummy_signal = torch.zeros(1, 1, sample_length)
            feat_out = self.branch_rp(dummy_signal)
            feat_dim = feat_out.view(1, -1).size(1)

        # Cabeza clasificadora sobre la concatenación (feat_dim * 2 -> 44800)
        self.fusion_head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim * 2, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        # x shape: (B, 2, L) -> Separar RP (canal 0) y AZ (canal 1)
        x_rp = x[:, 0:1, :]
        x_az = x[:, 1:2, :]

        feat_rp = torch.flatten(self.branch_rp(x_rp), start_dim=1)
        feat_az = torch.flatten(self.branch_az(x_az), start_dim=1)

        fused_features = torch.cat([feat_rp, feat_az], dim=1)
        return self.fusion_head(fused_features)


# -------------------------------------------------------------------------
# Obtener la longitud temporal real directamente del dataset
# -------------------------------------------------------------------------
sample_signal, _ = full_hc_dataset[0]
signal_len = sample_signal.shape[-1]
print(f"Longitud temporal real de la señal detectada: {signal_len} puntos.")


# -------------------------------------------------------------------------
# Evaluation helper using in-memory cached tensors
# -------------------------------------------------------------------------
def fast_eval(model, inputs_tensor, targets_tensor, batch_size=256, device="cuda"):
    model.eval()
    correct = 0
    total = targets_tensor.size(0)
    with torch.no_grad():
        for i in range(0, total, batch_size):
            x = inputs_tensor[i:i + batch_size].to(device)
            y = targets_tensor[i:i + batch_size].to(device)
            preds = model(x).argmax(dim=1)
            targets_idx = y.argmax(dim=1)
            correct += (preds == targets_idx).sum().item()
    return 100.0 * correct / total


# -------------------------------------------------------------------------
# Scratch Training on HC Subsets (train_20 to train_500) -> Evaluation on HC (Late Fusion)
# -------------------------------------------------------------------------

for seed in seeds:
    print(f"\n==========================================")
    print(f">>> Running HC Subsets Late Fusion (RP + AZ) for Seed: {seed}")
    print(f"==========================================")
    set_seed(seed)

    for s_name, n_samples in zip(subset_names, sample_sizes):
        print(f"\n--- Training on HC Subset: {s_name} ({n_samples} samples) ---")

        # 1. Extraer subconjunto balanceado desde HC T197
        train_subset_hc = get_balanced_transfer_subset(
            dataset=full_hc_dataset,
            total_samples=n_samples,
            seed=seed
        )
        batch_sz = min(32, n_samples)
        train_loader_hc = DataLoader(train_subset_hc, batch_size=batch_sz, shuffle=True)

        # 2. Inicializar modelo pasando la longitud real de la señal
        model = LateFusion1DCNN(sample_length=signal_len, num_classes=10).to(device)
        optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
        scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=25, gamma=0.1)

        # 3. Entrenar modelo
        model = train_model(
            model=model,
            loader=train_loader_hc,
            criterion=criterion,
            optimizer=optimizer,
            scheduler=scheduler,
            num_epochs=100,
            device=device
        )

        # 4. Evaluación directa sobre los espesores de prueba de HC (T185, T188, T191, T194)
        for vp in val_paths:
            acc = fast_eval(
                model=model,
                inputs_tensor=cached_val_tensors[vp]["inputs"],
                targets_tensor=cached_val_tensors[vp]["targets"],
                device=device
            )
            results[s_name][vp].append(acc)
            thickness_name = vp.split("/")[-2] if "/" in vp else vp.split("\\")[-2]
            print(f"[Seed {seed}] {s_name} -> HC_{thickness_name}: {acc:.2f}%")

## 5. Summary of Results - Fusion

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

# -------------------------------------------------------------------------
# Compile evaluation summary across evaluated HC sheet thicknesses (Late Fusion)
# -------------------------------------------------------------------------
summary_data = []

for s_name in subset_names:
    row = {"Subset": s_name}
    for vp in val_paths:
        accs = results[s_name][vp]
        mean_acc = np.mean(accs)
        std_acc = np.std(accs)

        # Extracción limpia y multiplataforma del nombre del espesor (e.g., 'T185')
        col_name = Path(vp).parent.name
        row[col_name] = f"{mean_acc:.2f} ± {std_acc:.2f}"
    summary_data.append(row)

# Orden de columnas para HC -> HC (evaluación cross-thickness excluyendo el set de entrenamiento T197)
column_order = ["Subset", "T185", "T188", "T191", "T194"]
df_late_fusion = pd.DataFrame(summary_data)[column_order]

# Exportar resultados consolidados a CSV
output_csv = "wear_LateFusion_HC_train_HC_eval_summary.csv"
df_late_fusion.to_csv(output_csv, index=False)

print("Consolidated Multimodal Late Fusion (RP + AZ) results (HC Train -> HC Evaluation):")
display(df_late_fusion)
print(f"Results successfully saved to: {output_csv}")